# 03 - Analisis de comparacion entre empresas

Este notebook desarrolla el foco 2 completo. Las preguntas centrales son:

- que empresas dominan el dataset por volumen;
- que fabricantes muestran mejor reputacion promedio;
- que tan consistente es la calidad interna de cada empresa;
- quien fabrica mejor una misma composicion;
- como se cruza seguridad, especializacion y escala.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import pandas as pd

from src.load_data import load_medicine_data
from src.enfoque_02_comparacion_empresas.analysis import (
    composition_company_comparison,
    composition_winners,
    manufacturer_consistency,
    manufacturer_reputation_ranking,
    manufacturer_side_effect_summary,
    manufacturer_specialization,
    market_share_proxy,
    plot_quality_vs_quantity,
    plot_reputation_ranking,
    plot_review_balance_boxplot,
    plot_specialization_heatmap,
    plot_top_manufacturers,
    quality_quantity_balance,
    top_medicines_by_manufacturer,
)
from src.enfoque_02_comparacion_empresas.cleaning import (
    COMPANY_COMPARISON_CLEAN_PATH,
    clean_company_comparison_data,
    load_company_comparison_clean_data,
)

plt.style.use("default")
pd.set_option("display.max_colwidth", 120)

try:
    df = load_company_comparison_clean_data(COMPANY_COMPARISON_CLEAN_PATH)
    print(f"Dataset procesado cargado desde: {COMPANY_COMPARISON_CLEAN_PATH}")
except FileNotFoundError:
    print("No existe CSV procesado. Se reconstruye desde raw...")
    df = clean_company_comparison_data(load_medicine_data(download_if_missing=False), save=False)

MIN_MEDICINES = 10
print(f"Shape analitico: {df.shape}")
print(f"Filtro principal para ranking: empresas con >= {MIN_MEDICINES} medicamentos")

## 1. Share de mercado (proxy)

No mide mercado real, pero si la **presencia relativa dentro del dataset**.
Sirve para separar empresas masivas de empresas de nicho.

In [ ]:
share_proxy = market_share_proxy(df, top_n=15)
share_proxy

In [ ]:
plot_top_manufacturers(df, top_n=15)
plt.show()

## 2. Ranking de reputacion por empresa

Usamos `review_balance = Excellent - Poor` como indicador central. Mientras mas
alto, mejor mezcla de reviews tiene la empresa.

In [ ]:
ranking = manufacturer_reputation_ranking(df, min_medicines=MIN_MEDICINES)
ranking.head(15)

In [ ]:
plot_reputation_ranking(df, top_n=15, min_medicines=MIN_MEDICINES)
plt.show()

## 3. Distribucion y consistencia interna

Una empresa puede tener un buen promedio pero ser muy irregular. Por eso miramos
la dispersion interna del `review_balance`.

In [ ]:
plot_review_balance_boxplot(df, top_n=12)
plt.show()

In [ ]:
consistency = manufacturer_consistency(df, min_medicines=MIN_MEDICINES)
print("Empresas mas consistentes (menor desviacion):")
display(consistency.head(10))

print("\nEmpresas mas irregulares (mayor desviacion):")
display(consistency.sort_values("review_balance_std", ascending=False).head(10))

## 4. Misma composicion, distinta empresa

Este es el insight mas fuerte del foco 2. Comparamos empresas que fabrican la
misma combinacion de ingredientes activos.

In [ ]:
comparison = composition_company_comparison(df, min_manufacturers=2)
comparison.head(15)

In [ ]:
winners = composition_winners(df, min_manufacturers=2)
winners.head(15)

In [ ]:
composition_gap = (
    comparison.groupby("composition_key")
    .agg(
        n_manufacturers=("manufacturer_clean", "nunique"),
        best_balance=("review_balance_mean", "max"),
        worst_balance=("review_balance_mean", "min"),
        total_medicines=("n_medicines", "sum"),
    )
)
composition_gap["gap_balance"] = composition_gap["best_balance"] - composition_gap["worst_balance"]
composition_gap.sort_values(["gap_balance", "n_manufacturers"], ascending=False).head(15)

## 5. Efectos secundarios por empresa

Aqui cruzamos seguridad con reputacion. Una empresa puede tener buen volumen,
pero tambien mas efectos secundarios promedio.

In [ ]:
side_summary = manufacturer_side_effect_summary(df, min_medicines=MIN_MEDICINES)
side_summary.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(
    side_summary["avg_side_effects"],
    side_summary["poor_mean"],
    s=side_summary["n_medicines"] * 4,
    alpha=0.7,
    color="#E15759",
    edgecolors="white",
)
ax.set_title("Seguridad vs malas reviews por empresa", fontsize=13, fontweight="bold")
ax.set_xlabel("Promedio de side effects por medicamento")
ax.set_ylabel("Poor Review % promedio")
ax.grid(linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Especializacion por empresa

Como `Uses` venia muy sucio, se transformo a areas terapeuticas amplias. Esto
permite detectar nichos de negocio sin depender del texto raw.

In [ ]:
specialization = manufacturer_specialization(df, min_medicines=MIN_MEDICINES)
specialization.sort_values(["share_within_company_pct", "n_medicines"], ascending=False).head(20)

In [ ]:
plot_specialization_heatmap(df, top_n_manufacturers=10)
plt.show()

In [ ]:
empresa_nicho = (
    specialization.sort_values(["manufacturer_clean", "share_within_company_pct"], ascending=[True, False])
    .groupby("manufacturer_clean", as_index=False)
    .head(1)
    .reset_index(drop=True)
)
empresa_nicho.head(15)

## 7. Balance calidad vs cantidad

Con esto distinguimos empresas:

- grandes y fuertes;
- grandes pero promedio;
- pequenas premium;
- pequenas e irregulares.

In [ ]:
plot_quality_vs_quantity(df, min_medicines=MIN_MEDICINES)
plt.show()

In [ ]:
balance = quality_quantity_balance(df, min_medicines=MIN_MEDICINES)
balance.head(20)

## 8. Top medicamentos por empresa

Esto ayuda a identificar el producto estrella de cada laboratorio dentro del dataset.

In [ ]:
top_medicines_by_manufacturer(df, min_company_size=MIN_MEDICINES, top_n=3).head(30)

## 9. Conclusiones del foco 2

- El dataset permite un ranking razonable de empresas si se filtra por volumen minimo.
- El promedio no basta: la consistencia interna cambia bastante entre empresas.
- La comparacion por `composition_key` es el insight mas fuerte del enfoque.
- Side effects y reputacion no siempre se mueven juntos, por lo que conviene mirar ambas dimensiones.
- La especializacion terapeutica ayuda a explicar por que dos empresas grandes pueden tener perfiles de calidad muy distintos.

Siguiente paso natural si quieres profundizar:

1. endurecer el filtro por volumen para trabajar solo con empresas robustas;
2. pasar de `composition_key` a composicion + dosis si quieres una comparacion mas estricta;
3. exportar tablas finales a `outputs/tables/` para la entrega.